In [1]:
from model.engine import ClaudeAdapter

In [2]:
model = ClaudeAdapter("anthropic.claude-3-5-sonnet-20240620-v1:0")

KeyboardInterrupt: 

In [ ]:
p = model.format_prompt("what is a cat")
ans = model.complete(p)[0]
ans

"Here's a basic definition of a cat:\n\n<answer>\nA cat is a small domesticated carnivorous mammal with soft fur, a short snout, and retractable claws. Cats are often kept as pets and are known for their independent nature, hunting skills, and ability to purr. Key features of cats include:\n\n1. Scientific name: Felis catus\n2. Part of the Felidae family, which includes lions, tigers, and other wild cats\n3. Typically have four legs, a tail, whiskers, and sharp teeth\n4. Excellent night vision and keen senses of smell and hearing\n5. Known for their grooming habits and agile, flexible bodies\n6. Come in many breeds with various coat colors and patterns\n7. Popular companion animals in many cultures around the world\n\nCats have been living alongside humans for thousands of years and are valued both as pets and for their historical role"

In [ ]:
def prompting(data):
    task_prompt = '''
    ### Role: you are a data labeller. you need to find whether the answer provided answers the question, and why it fails. If possible, please extract the excerpt where the answer come from in the provided source, else label the data as baseless
    1. for assessing whether answer answers question, the label you can use is 
    2. provide reason (following sample below). they need to be short.
    3. excerpt needs to be exact match from source. if no excerpt is found return None

    ### data:
    input: in json format
    format your answer in json.

    ### Example:
    here are some example input and output:
    input 1:
    {
        "question": 12618,
        "prompt": "In what country is Karimu?",
        "real_ans_list": ["Iran", "Islamic Republic of Iran", "Persia"],
        "source": [({'title': 'Karimu', 'context': "Title: Karimu\nKarimu (Persian: ÙƒØ±ÙŠÙ…Ùˆ) is a village in Masabi Rural District of Aysak District, Sarayan County, South Khorasan province, Iran.\n\n\n== Demographics ==\n\n\n=== Population ===\nAt the time of the 2006 National Census, the village's population was 801 in 287 households. The following census in 2011 counted 742 people in 289 households. The 2016 census measured the population of the village as 679 people in 282 households. It was the most populous village in its rural district.\n\n\n== See also ==\n Iran portal\n\n\n== Notes ==\n\n\n== References =="}, 0.7251806855201721), ({'title': 'Education in South Africa', 'context': "Title: Education in South Africa\nEducation in South Africa is governed by two national departments, namely the Department of Basic Education (DBE), which is responsible for primary and secondary schools, and the Department of Higher Education and Training (DHET), which is responsible for tertiary education and vocational training. Prior to 2009, both departments were represented in a single Department of Education. Among sub-Saharan African countries, South Africa has one of the highest literacy rates. According to The World Factbook - Central Intelligence Agency as of 2019, 95% of the population aged from  15 and over can read and write in South Africa were respectively literate.\nThe DBE department deals with public schools, private schools (also referred to by the department as independent schools), early childhood development (ECD) centres, and special needs schools. The public schools and private schools are collectively known as ordinary schools, which are roughly 97% of schools in South Africa. Unlike in most countries, many public schools charge tuition (referred to as fees). No-fee schools were introduced on a limited basis in 2007.\nThe DHET department deals with further education and training (FET) colleges now known as Technical and Vocational Education and Training (TVET) colleges, adult basic education and training (ABET) centres, and higher education (HE) institutions.\nThe nine provinces of South Africa also have their own education departments that are responsible for implementing the policies of the national department and dealing with local issues.\nIn 2010, the basic.."
        "answer": "Sierra Leone."
    }
    

    output 1:
    {
        "question": 12618,
        "wrong_reason": used wrong source,
        notes: "was asking for country of a place, gave birthplace of person with same surname",
        "source excerpt": "{'title': 'John Karimu', 'context': 'Title: John Karimu\nJohn Arouna Karimu is a Mende hailed from Daru Village, Kailahun District, in the Eastern Region of Sierra Leone."
    }
    as you see the real answer should be Iran (from source)
    
    input 2:
    {
        'question': 2274,
        'prompt': 'What filmmaker was responsible for Second Chances?',
        'real_ans_list': '["LeVar Burton"]',
        'source': '[({\'title\': \'Second Chances (Star Trek: The Next Generation)\', \'context\': \'Title: Second Chances (Star Trek: The Next Generation)\\n"Second Chances" is the 150th episode of the American syndicated science fiction television series Star Trek: The Next Generat
        "answer": 'James Fargo.'
    }
    
    output 2:
    {
        "question": 2274,
        "wrong_reason": "confusing object",
        notes: "the answer likely refers to another movie second chance. since there are multiple instance of second chance in source it is not clear which one it is referring to.",
        "source excerpt": "Second Chances is a 1998 film directed by James Fargo. It stars Tom Amandes, former A Little Princess star Kelsey Mulrooney"
    }
    as you see the real answer should be Iran (from source)
    '''
    return f"{task_prompt}\n{data}"


In [ ]:
import pandas as pd
import json

In [ ]:
path = "/homes/lst20/fyp/fyp_resources/LexEval-main/analysis/df_para_.csv"
df_para = pd.read_csv(path)

res_str_para = []
index_to_drop = df_para[(df_para["base_found_rag_match"] == False) & (df_para["type"] == "RootNode")].index
# Step 2: Drop the collected rows from the DataFrame
df_para = df_para.drop(index_to_drop)
df_para = df_para[~df_para["base_found_rag_match"]]

for i in range(40, 80, 5):
    row = dict(df_para.iloc[i])
    data = {
        'question': int(row['question_id']),
        'prompt': row['prompt'],
        "real_ans_list": row['possible_answers'],
        'source': str(row['rag_closest_match']),
        'answer': row['base_rag']
    }

    data_json_str = json.dumps(data, indent=2)

    p = model.format_prompt(prompting(data_json_str))
    ans = model.complete(p)[0]
    res_str_para.append((row['layer'], ans))
    print((row['layer'], ans))

(1, '{\n  "question": 1605,\n  "wrong_reason": "partially correct but incomplete",\n  "notes": "The answer correctly states Dawn\'s relationship to Buffy but omits information about her mother Joyce Summers, which is part of her parentage.",\n  "source excerpt": "Dawn Marie Summers is first introduced as Buffy\'s (Sarah Michelle Gellar) younger sister at the end of Buffy season 5 premiere \\"Buffy vs. Dracula\\", though Buffy had been previously established as an only child. Initially, the mystery of Dawn\'s sudden existence is not acknowledged in the series, with the other characters accepting her as a part of the status quo. Four episodes later, Buffy discovers Dawn is, in fact, a mystical object known as The Key; a group of monks transformed The Key into human form and sent it to the Slayer for protection from the villain')
(3, '{\n  "question": 1605,\n  "wrong_reason": "incorrect information",\n  "notes": "The answer provided is incorrect. Joyce Summers is Dawn\'s birth mother, not

In [ ]:
print(res_str_para)

In [ ]:
path = "/homes/lst20/fyp/fyp_resources/LexEval-main/analysis/df_paraprefix_.csv"
df_paraprefix_ = pd.read_csv(path)

res_str_paraprefix = []
index_to_drop = df_paraprefix_[(df_paraprefix_["base_found_rag_match"] == False) & (df_paraprefix_["type"] == "RootNode")].index
# Step 2: Drop the collected rows from the DataFrame
df_paraprefix_ = df_paraprefix_.drop(index_to_drop)
df_paraprefix_ = df_paraprefix_[~df_paraprefix_["base_found_rag_match"]]

for i in range(20):
    row = dict(df_paraprefix_.iloc[i])
    data = {
        'question': int(row['question_id']),
        'prompt': row['prompt'],
        "real_ans_list": row['possible_answers'],
        'source': str(row['rag_closest_match']),
        'answer': row['base_rag']
    }

    data_json_str = json.dumps(data, indent=2)

    p = model.format_prompt(prompting(data_json_str))
    ans = model.complete(p)[0]
    res_str_paraprefix.append((row['layer'], ans))
    print((row['layer'], ans))

print(res_str_paraprefix)

In [ ]:
res_str_paraprefix